© 2026 by Tamás Takács is licensed under CC BY-NC-SA 4.0. To view a copy of this license, visit https://creativecommons.org/licenses/by-nc-sa/4.0/

English translation managed by Tamás Takács. The translation was produced with AI assistance.

# HAIO 2025 - University Admission Solution

**Hungarian AI Olympiad - Online Qualifier 2025**

Task: Predicting university admission outcomes (binary classification)  
Evaluation metric: **ROC-AUC**  
Kaggle competition: https://www.kaggle.com/competitions/magyar-mi-diakolimpia-online-valogato-2025

## 0. Installation and imports

In [ ]:
!pip install lightgbm xgboost geopandas --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, classification_report
from sklearn.model_selection import GridSearchCV

import xgboost as xgb
import lightgbm as lgb

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

RANDOM_STATE = 42
print('Imports successful!')

## 1. Exploratory Data Analysis (EDA)

### 1.1 Loading the data

In [ ]:
# Loading the data
train = pd.read_csv('../adatok/train.csv')
test = pd.read_csv('../adatok/test.csv')

print(f'Train size: {train.shape}')
print(f'Test size:  {test.shape}')
print(f'\nColumns ({len(train.columns)} total):')
print(train.columns.tolist())

In [ ]:
train.head(10)

In [ ]:
train.dtypes

In [ ]:
train.describe()

### 1.2 Examining missing values and -1

In [ ]:
# Genuine NaN values
print('=== Genuine NaN values ===')
print(train.isnull().sum())

print('\n=== Number of -1 values per column ===')
# The -1 values are "missing/not applicable" markers (e.g. did not take the subject)
minus_one_counts = (train == -1).sum()
print(minus_one_counts[minus_one_counts > 0])

### 1.3 Distribution of the target variable

In [ ]:
TARGET = 'Felvételi Eredmény'

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
target_counts = train[TARGET].value_counts()
colors = ['#e74c3c', '#2ecc71']
axes[0].bar(target_counts.index.astype(int), target_counts.values, color=colors)
axes[0].set_xlabel('Admission Outcome')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of the target variable')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Rejected (0)', 'Admitted (1)'])

for i, v in enumerate(target_counts.values):
    axes[0].text(target_counts.index[i], v + 30, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(target_counts.values, labels=['Rejected (0)', 'Admitted (1)'],
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Target variable proportions')

plt.tight_layout()
plt.show()

print(f'Admission rate: {train[TARGET].mean():.3f}')

### 1.4 Visualizing important features

In [ ]:
# Distribution of grades by the target variable
osztalyzatok = ['Osztályzat_9', 'Osztályzat_10', 'Osztályzat_11', 'Osztályzat_12']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, col in enumerate(osztalyzatok):
    ax = axes[i // 2][i % 2]
    for label, color in zip([0, 1], colors):
        subset = train[train[TARGET] == label][col]
        ax.hist(subset, bins=20, alpha=0.6, color=color,
                label=f'Outcome={int(label)}')
    ax.set_title(col)
    ax.set_xlabel('Grade')
    ax.set_ylabel('Count')
    ax.legend()

plt.suptitle('Distribution of grades by admission outcome', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Final exam subject scores
tantargyak = ['Történelem', 'Matematika', 'Magyar Nyelv és Irodalom',
              'Informatika', 'Biológia', 'Fizika', 'Angol', 'Német']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for i, col in enumerate(tantargyak):
    ax = axes[i // 4][i % 4]
    # We only plot the values that are not -1
    for label, color in zip([0, 1], colors):
        subset = train[(train[TARGET] == label) & (train[col] != -1)][col]
        ax.hist(subset, bins=25, alpha=0.6, color=color,
                label=f'Outcome={int(label)}')
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('Final exam subject scores by admission outcome', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap for the numeric columns
numerikus_oszlopok = train.select_dtypes(include=[np.number]).columns.tolist()
# We leave out the ID
numerikus_oszlopok = [c for c in numerikus_oszlopok if c != 'ID']

corr = train[numerikus_oszlopok].corr()

fig, ax = plt.subplots(figsize=(18, 14))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0,
            annot=True, fmt='.2f', square=True, linewidths=0.5,
            ax=ax, annot_kws={'size': 7})
ax.set_title('Correlation heatmap', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation with the target variable
target_corr = corr[TARGET].drop(TARGET).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
bar_colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in target_corr.values]
target_corr.plot(kind='barh', ax=ax, color=bar_colors)
ax.set_title('Correlation with the Admission Outcome', fontsize=14)
ax.set_xlabel('Pearson correlation')
ax.axvline(x=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

### 1.5 Visualization by county

In [ ]:
# Admission rate by county
varmegye_stats = train.groupby('Vármegye')[TARGET].agg(['mean', 'count']).reset_index()
varmegye_stats.columns = ['Vármegye', 'Felvételi arány', 'Létszám']
varmegye_stats = varmegye_stats.sort_values('Felvételi arány', ascending=True)

print(varmegye_stats.to_string(index=False))

In [ ]:
# Map visualization (if geopandas is available)
try:
    import geopandas as gpd

    gdf = gpd.read_file('../adatok/counties.geojson')

    # Identifying the name column in the GeoJSON
    name_col = None
    for col in gdf.columns:
        if col.lower() in ['name', 'név', 'nev', 'megye', 'varmegye', 'admin_name']:
            name_col = col
            break
    if name_col is None:
        # We use the first non-geometry column
        name_col = [c for c in gdf.columns if c != 'geometry'][0]

    print(f'GeoJSON name column: {name_col}')
    print(f'GeoJSON values: {sorted(gdf[name_col].unique())}')
    print(f'Train Vármegye values: {sorted(train["Vármegye"].unique())}')

    # Merging
    merged = gdf.merge(varmegye_stats, left_on=name_col, right_on='Vármegye', how='left')

    fig, ax = plt.subplots(figsize=(14, 10))
    merged.plot(column='Felvételi arány', ax=ax, legend=True,
                cmap='RdYlGn', edgecolor='black', linewidth=0.5,
                legend_kwds={'label': 'Admission rate', 'shrink': 0.6})
    ax.set_title('Admission rate by county', fontsize=16)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f'Map visualization failed ({e}), a bar chart follows:')

    # Fallback bar chart
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.barh(varmegye_stats['Vármegye'], varmegye_stats['Felvételi arány'],
            color=plt.cm.RdYlGn(varmegye_stats['Felvételi arány']))
    ax.set_xlabel('Admission rate')
    ax.set_title('Admission rate by county', fontsize=14)
    ax.axvline(x=train[TARGET].mean(), color='red', linestyle='--',
               label=f'Mean: {train[TARGET].mean():.3f}')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 2. Preprocessing

In [ ]:
# Copy of the original data
df_train = train.copy()
df_test = test.copy()

# Separating the target variable and the ID
y = df_train[TARGET].astype(int)
train_ids = df_train['ID']
test_ids = df_test['ID']

df_train = df_train.drop(columns=[TARGET, 'ID'])
df_test = df_test.drop(columns=['ID'])

print(f'Number of features: {df_train.shape[1]}')
print(f'Train: {df_train.shape[0]}, Test: {df_test.shape[0]}')

### 2.1 Feature engineering

In [ ]:
def feature_engineering(df):
    """Creating features from the raw data."""
    df = df.copy()

    # --- Grades ---
    osztalyzat_cols = ['Osztályzat_9', 'Osztályzat_10', 'Osztályzat_11', 'Osztályzat_12']
    df['Osztályzat_átlag'] = df[osztalyzat_cols].mean(axis=1)
    df['Osztályzat_trend'] = df['Osztályzat_12'] - df['Osztályzat_9']  # improvement/decline
    df['Osztályzat_max'] = df[osztalyzat_cols].max(axis=1)
    df['Osztályzat_min'] = df[osztalyzat_cols].min(axis=1)
    df['Osztályzat_szórás'] = df[osztalyzat_cols].std(axis=1)

    # --- Final exam subjects (where -1 = did not take it) ---
    tantargyak = ['Történelem', 'Matematika', 'Magyar Nyelv és Irodalom',
                  'Informatika', 'Biológia', 'Fizika', 'Angol', 'Német']

    # Number of subjects (those not equal to -1)
    tantargy_mask = df[tantargyak].replace(-1, np.nan)
    df['Tantárgyak_száma'] = tantargy_mask.notna().sum(axis=1)
    df['Tantárgy_átlag'] = tantargy_mask.mean(axis=1)
    df['Tantárgy_max'] = tantargy_mask.max(axis=1)
    df['Tantárgy_min'] = tantargy_mask.min(axis=1)
    df['Tantárgy_szórás'] = tantargy_mask.std(axis=1)

    # --- Advanced-level final exams ---
    emelt_cols = ['Informatika_emelt', 'Biológia_emelt', 'Fizika_emelt',
                  'Angol_emelt', 'Német_emelt', 'Matematika_emelt',
                  'Történelem_emelt', 'Magyar Nyelv és Irodalom_emelt']

    # We treat the -1 values as 0 (subject not relevant)
    emelt_valid = df[emelt_cols].replace(-1, 0)
    df['Emelt_száma'] = emelt_valid.sum(axis=1)

    # --- Combined features ---
    df['Osztályzat_x_Presztízs'] = df['Osztályzat_átlag'] * df['Középiskola Presztízse']
    df['Tanulás_x_Osztályzat'] = df['Tanulási Szokások'] * df['Osztályzat_átlag']
    df['Extra_és_Verseny'] = df['Extrakurrikuláris Tevékenységek'] + df['Versenyeken Való Részvétel']
    df['Ajánlás_és_Munka'] = df['Ajánlások Száma'] + df['Munkatapasztalat']

    # --- Encoding the county (Label Encoding) ---
    # Collecting all possible counties from train and test
    le = LabelEncoder()
    df['Vármegye_kód'] = le.fit_transform(df['Vármegye'].astype(str))

    return df, le


# Processing train and test together for consistent encoding
df_combined = pd.concat([df_train, df_test], axis=0, ignore_index=True)
df_combined, varmegye_encoder = feature_engineering(df_combined)

# Splitting them apart
df_train_fe = df_combined.iloc[:len(df_train)].copy()
df_test_fe = df_combined.iloc[len(df_train):].copy()

# Removing the textual county column
df_train_fe = df_train_fe.drop(columns=['Vármegye'])
df_test_fe = df_test_fe.drop(columns=['Vármegye'])

print(f'Number of features after feature engineering: {df_train_fe.shape[1]}')
print(f'New features: {[c for c in df_train_fe.columns if c not in df_train.columns]}')

### 2.2 Handling the -1 values

In [ ]:
# We keep the -1 values for the tree-based models (they can handle them),
# but for the linear models we will replace them with NaN.
# For now we also create an indicator column.

tantargyak = ['Informatika', 'Biológia', 'Fizika', 'Angol', 'Német']
for col in tantargyak:
    if col in df_train_fe.columns:
        df_train_fe[f'{col}_hiányzik'] = (df_train_fe[col] == -1).astype(int)
        df_test_fe[f'{col}_hiányzik'] = (df_test_fe[col] == -1).astype(int)

print(f'Final number of features: {df_train_fe.shape[1]}')
print(df_train_fe.head())

### 2.3 Train/validation split

In [ ]:
X = df_train_fe.values
feature_names = df_train_fe.columns.tolist()
X_test_final = df_test_fe.values

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape}, Validation: {X_val.shape}')
print(f'Train target variable distribution: {np.mean(y_train):.3f}')
print(f'Validation target variable distribution: {np.mean(y_val):.3f}')

## 3. Baseline model - Logistic Regression

In [ ]:
# Scaling is required for logistic regression
scaler = StandardScaler()

# Handling NaNs coming from the -1 values (for the linear model)
X_train_lr = pd.DataFrame(X_train, columns=feature_names).copy()
X_val_lr = pd.DataFrame(X_val, columns=feature_names).copy()

# Replace -1 values with 0 for the linear model
X_train_lr = X_train_lr.replace(-1, 0)
X_val_lr = X_val_lr.replace(-1, 0)

# Fill NaN values with 0
X_train_lr = X_train_lr.fillna(0)
X_val_lr = X_val_lr.fillna(0)

X_train_scaled = scaler.fit_transform(X_train_lr)
X_val_scaled = scaler.transform(X_val_lr)

# Logistic regression
lr_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, C=1.0)
lr_model.fit(X_train_scaled, y_train)

# Prediction and evaluation
y_pred_lr = lr_model.predict_proba(X_val_scaled)[:, 1]
auc_lr = roc_auc_score(y_val, y_pred_lr)
print(f'Logistic Regression ROC-AUC: {auc_lr:.4f}')

In [ ]:
# ROC curve
fpr_lr, tpr_lr, _ = roc_curve(y_val, y_pred_lr)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr_lr, tpr_lr, 'b-', linewidth=2,
        label=f'Logistic Regression (AUC = {auc_lr:.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.5)')
ax.set_xlabel('False Positive Rate (FPR)')
ax.set_ylabel('True Positive Rate (TPR)')
ax.set_title('ROC curve - Baseline model', fontsize=14)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Fejlettebb modellek

### 4.1 Random Forest

In [ ]:
# Random Forest - hyperparameter search
rf_params = {
    'n_estimators': [200, 500],
    'max_depth': [8, 12, 16],
    'min_samples_split': [5, 10],
    'min_samples_leaf': [2, 4],
}

rf_model = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

rf_grid = GridSearchCV(
    rf_model, rf_params,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

# Tree-based models can handle -1 values, but not NaNs
X_train_filled = np.nan_to_num(X_train, nan=0.0)
X_val_filled = np.nan_to_num(X_val, nan=0.0)

rf_grid.fit(X_train_filled, y_train)

print(f'\nBest RF parameters: {rf_grid.best_params_}')
print(f'Best RF CV ROC-AUC: {rf_grid.best_score_:.4f}')

y_pred_rf = rf_grid.best_estimator_.predict_proba(X_val_filled)[:, 1]
auc_rf = roc_auc_score(y_val, y_pred_rf)
print(f'RF Validation ROC-AUC: {auc_rf:.4f}')

### 4.2 XGBoost

In [ ]:
# XGBoost hyperparameter search
xgb_params = {
    'n_estimators': [200, 500],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'reg_alpha': [0, 0.1],
    'reg_lambda': [1, 5],
}

xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    use_label_encoder=False
)

xgb_grid = GridSearchCV(
    xgb_model, xgb_params,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(X_train_filled, y_train)

print(f'\nBest XGBoost parameters: {xgb_grid.best_params_}')
print(f'Best XGBoost CV ROC-AUC: {xgb_grid.best_score_:.4f}')

y_pred_xgb = xgb_grid.best_estimator_.predict_proba(X_val_filled)[:, 1]
auc_xgb = roc_auc_score(y_val, y_pred_xgb)
print(f'XGBoost Validation ROC-AUC: {auc_xgb:.4f}')

### 4.3 LightGBM

In [ ]:
# LightGBM hyperparameter search
lgb_params = {
    'n_estimators': [200, 500, 1000],
    'max_depth': [4, 6, 8, -1],
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [31, 63],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'reg_alpha': [0, 0.1],
    'reg_lambda': [0, 1],
}

lgb_model = lgb.LGBMClassifier(
    objective='binary',
    metric='auc',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

lgb_grid = GridSearchCV(
    lgb_model, lgb_params,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

lgb_grid.fit(X_train_filled, y_train)

print(f'\nBest LightGBM parameters: {lgb_grid.best_params_}')
print(f'Best LightGBM CV ROC-AUC: {lgb_grid.best_score_:.4f}')

y_pred_lgb = lgb_grid.best_estimator_.predict_proba(X_val_filled)[:, 1]
auc_lgb = roc_auc_score(y_val, y_pred_lgb)
print(f'LightGBM Validation ROC-AUC: {auc_lgb:.4f}')

### 4.4 Comparing the models

In [ ]:
# Comparison table
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost', 'LightGBM'],
    'Validation ROC-AUC': [auc_lr, auc_rf, auc_xgb, auc_lgb],
    'CV ROC-AUC': ['-', f'{rf_grid.best_score_:.4f}',
                   f'{xgb_grid.best_score_:.4f}', f'{lgb_grid.best_score_:.4f}']
}).sort_values('Validation ROC-AUC', ascending=False)

print(results.to_string(index=False))

# Comparison of the ROC curves
fpr_rf, tpr_rf, _ = roc_curve(y_val, y_pred_rf)
fpr_xgb, tpr_xgb, _ = roc_curve(y_val, y_pred_xgb)
fpr_lgb, tpr_lgb, _ = roc_curve(y_val, y_pred_lgb)

fig, ax = plt.subplots(figsize=(10, 10))
ax.plot(fpr_lr, tpr_lr, linewidth=2, label=f'Logistic Regression (AUC = {auc_lr:.4f})')
ax.plot(fpr_rf, tpr_rf, linewidth=2, label=f'Random Forest (AUC = {auc_rf:.4f})')
ax.plot(fpr_xgb, tpr_xgb, linewidth=2, label=f'XGBoost (AUC = {auc_xgb:.4f})')
ax.plot(fpr_lgb, tpr_lgb, linewidth=2, label=f'LightGBM (AUC = {auc_lgb:.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.5)')
ax.set_xlabel('False Positive Rate (FPR)', fontsize=12)
ax.set_ylabel('True Positive Rate (TPR)', fontsize=12)
ax.set_title('Comparison of ROC curves', fontsize=16)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Final model and submission

### 5.1 Selecting the best model and retraining it on the full training data

In [ ]:
# Selecting the best model
best_models = {
    'Random Forest': (rf_grid.best_estimator_, auc_rf),
    'XGBoost': (xgb_grid.best_estimator_, auc_xgb),
    'LightGBM': (lgb_grid.best_estimator_, auc_lgb),
}

best_name = max(best_models, key=lambda k: best_models[k][1])
best_model_template = best_models[best_name][0]
print(f'Best model: {best_name} (AUC = {best_models[best_name][1]:.4f})')
print(f'Parameters: {best_model_template.get_params()}')

In [ ]:
# Retraining on the full training data
X_full = np.nan_to_num(X, nan=0.0)
X_test_final_filled = np.nan_to_num(X_test_final, nan=0.0)

# Clone of the best model, trained on the full data
from sklearn.base import clone
final_model = clone(best_model_template)
final_model.fit(X_full, y)

# Cross-validation on the full data
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(clone(best_model_template), X_full, y,
                            cv=cv, scoring='roc_auc', n_jobs=-1)

print(f'\n5-fold CV ROC-AUC values: {cv_scores}')
print(f'Mean CV ROC-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

### 5.2 Feature importance

In [ ]:
# Visualizing feature importances
if hasattr(final_model, 'feature_importances_'):
    importances = final_model.feature_importances_
    feat_imp = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values('Importance', ascending=True)

    # Top 20 features
    top_n = min(20, len(feat_imp))
    feat_imp_top = feat_imp.tail(top_n)

    fig, ax = plt.subplots(figsize=(12, 10))
    bars = ax.barh(feat_imp_top['Feature'], feat_imp_top['Importance'],
                   color=plt.cm.viridis(np.linspace(0.2, 0.8, top_n)))
    ax.set_xlabel('Importance', fontsize=12)
    ax.set_title(f'Top {top_n} most important features ({best_name})', fontsize=14)
    plt.tight_layout()
    plt.show()

    # Full list
    print('\nImportance of all features:')
    print(feat_imp.sort_values('Importance', ascending=False).to_string(index=False))

### 5.3 Generating the submission file

In [ ]:
# Prediction on the test data
y_test_pred = final_model.predict(X_test_final_filled)

# Submission DataFrame
submission = pd.DataFrame({
    'ID': test_ids,
    'Admission Result': y_test_pred.astype(int)
})

# Check
print(f'Submission size: {submission.shape}')
print(f'Distribution of the predicted values:')
print(submission['Admission Result'].value_counts())
print(f'Admission rate (predicted): {submission["Admission Result"].mean():.3f}')
print(f'Admission rate (train): {y.mean():.3f}')

submission.head(10)

In [ ]:
# Saving the submission
submission.to_csv('submission.csv', index=False)
print('submission.csv saved successfully!')
print(f'\n--- Final result ---')
print(f'Model: {best_name}')
print(f'5-fold CV ROC-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')